In [2]:
import duckdb
import pandas as pd

# File path to your CSV
csv_file = 'vgchartz_cleaned(Pandas).csv'

In [3]:
# Top 10 Sales Worldwide
kpi1 = duckdb.sql(f"""
    SELECT 
        title, 
        console, 
        publisher, 
        total_sales
    FROM '{csv_file}'
    WHERE total_sales IS NOT NULL
    ORDER BY total_sales DESC
    LIMIT 10
""").df()

kpi1

,title,console,publisher,total_sales
0,Grand Theft Auto V,PS3,Rockstar Games,20.32
1,Grand Theft Auto V,PS4,Rockstar Games,19.39
2,Grand Theft Auto: Vice City,PS2,Rockstar Games,16.15
3,Grand Theft Auto V,X360,Rockstar Games,15.86
4,Call of Duty: Black Ops 3,PS4,Activision,15.09
5,Call of Duty: Modern Warfare 3,X360,Activision,14.82
6,Call of Duty: Black Ops,X360,Activision,14.74
7,Red Dead Redemption 2,PS4,Rockstar Games,13.94
8,Call of Duty: Black Ops II,X360,Activision,13.86
9,Call of Duty: Black Ops II,PS3,Activision,13.80


In [4]:
# Peak Industry Year & Yearly Growth Trend
kpi2 = duckdb.sql(f"""
    SELECT 
        YEAR(CAST(release_date AS DATE)) AS release_year,
        COUNT(title) AS games_released,
        ROUND(SUM(total_sales), 2) AS yearly_global_sales,
        ROUND(AVG(total_sales), 2) AS avg_sales_per_game
    FROM '{csv_file}'
    WHERE release_date IS NOT NULL 
      AND total_sales IS NOT NULL
    GROUP BY release_year
    ORDER BY release_year ASC
""").df()

kpi2

,release_year,games_released,yearly_global_sales,avg_sales_per_game
0,1977,3,2.50,0.83
1,1978,7,2.36,0.34
2,1979,1,0.31,0.31
3,1980,5,2.26,0.45
4,1981,6,7.73,1.29
5,1982,44,28.99,0.66
6,1983,37,22.68,0.61
7,1984,10,4.85,0.49
8,1985,4,2.19,0.55
9,1986,11,10.35,0.94


In [19]:
# Hardware Ecosystem Specialization by Genre
kpi3 = duckdb.sql(f"""
    SELECT 
        CASE 
            WHEN console IN ('PS', 'PS2', 'PS3', 'PS4', 'PS5', 'PSP', 'PSV') THEN 'Sony PlayStation'
            WHEN console IN ('XB', 'X360', 'XOne', 'XSX') THEN 'Microsoft Xbox'
            WHEN console IN ('N64', 'GC', 'Wii', 'WiiU', 'NS', 'GB', 'GBA', 'DS', '3DS') THEN 'Nintendo'
            WHEN console = 'PC' THEN 'PC'
            ELSE 'Other Hardware'
        END AS hardware_ecosystem,
        genre,
        COUNT(title) AS total_games,
        ROUND(SUM(total_sales), 2) AS genre_sales
    FROM '{csv_file}'
    GROUP BY hardware_ecosystem, genre
    ORDER BY hardware_ecosystem, genre_sales DESC
""").df()

print(kpi3.to_string())

   hardware_ecosystem             genre  total_games  genre_sales
0      Microsoft Xbox           Shooter          614       368.61
1      Microsoft Xbox            Action          771       250.00
2      Microsoft Xbox            Sports          599       244.11
3      Microsoft Xbox            Racing          400       108.74
4      Microsoft Xbox      Role-Playing          336        91.40
5      Microsoft Xbox              Misc          395        76.08
6      Microsoft Xbox          Fighting          228        54.77
7      Microsoft Xbox  Action-Adventure          249        45.66
8      Microsoft Xbox         Adventure          367        38.94
9      Microsoft Xbox          Platform          219        30.85
10     Microsoft Xbox        Simulation          163        21.91
11     Microsoft Xbox             Music           52        13.96
12     Microsoft Xbox          Strategy          134        10.04
13     Microsoft Xbox            Puzzle           91         2.54
14     Mic

In [6]:
# Regional Hits vs. Flops (Popular in Japan, but low NA sales)
kpi4 = duckdb.sql(f"""
    SELECT 
        title, 
        console, 
        genre, 
        jp_sales, 
        na_sales,
        pal_sales
    FROM '{csv_file}'
    WHERE jp_sales >= 1.0 
    ORDER BY jp_sales DESC
    LIMIT 10
""").df()

# could add "AND (na_sales < 0.1 OR na_sales IS NULL)""

print(kpi4.to_string())

                                                   title console         genre  jp_sales  na_sales  pal_sales
0                                         Hot Shots Golf      PS        Sports      2.13      0.29       0.20
1                            Famista '89 - Kaimaku Han!!     NES        Sports      2.05       NaN        NaN
2                                        R.B.I. Baseball     NES        Sports      2.05      0.15        NaN
3                                     Final Fantasy XIII     PS3  Role-Playing      1.87      1.75       1.23
4                                        Dragon Quest XI     3DS  Role-Playing      1.82       NaN        NaN
5                                        Super Puyo Puyo    SNES        Puzzle      1.69       NaN        NaN
6                         Tomodachi Collection: New Life     3DS    Simulation      1.69       NaN        NaN
7  Ninja Hattori Kun: Ninja wa Shuugyou Degogiru no Maki     NES      Platform      1.50       NaN        NaN
8         

In [7]:
# Critical Acclaim vs. Commercial Success
kpi5 = duckdb.sql(f"""
    SELECT 
        CASE 
            WHEN critic_score >= 9.0 THEN '9.0+ (Masterpiece)'
            WHEN critic_score >= 8.0 THEN '8.0-8.9 (Great)'
            WHEN critic_score >= 7.0 THEN '7.0-7.9 (Good)'
            WHEN critic_score >= 5.0 THEN '5.0-6.9 (Average)'
            ELSE 'Under 5.0 (Poor)'
        END AS review_tier,
        COUNT(title) AS total_games,
        ROUND(AVG(total_sales), 2) AS avg_global_sales,
        ROUND(MAX(total_sales), 2) AS top_selling_game_in_tier
    FROM '{csv_file}'
    WHERE critic_score IS NOT NULL 
      AND total_sales IS NOT NULL
    GROUP BY review_tier
    ORDER BY avg_global_sales DESC
""").df()

print(kpi5)

          review_tier  total_games  avg_global_sales  top_selling_game_in_tier
0  9.0+ (Masterpiece)          279              2.08                     20.32
1     8.0-8.9 (Great)         1050              1.10                     15.09
2      7.0-7.9 (Good)         1188              0.58                     10.13
3   5.0-6.9 (Average)         1244              0.42                     10.41
4    Under 5.0 (Poor)          350              0.27                      4.06


In [18]:
# Regional Market Share & Preference Index
kpi6 = duckdb.sql(f"""
    SELECT 
        genre,
        ROUND(SUM(na_sales), 2) AS na_total,
        ROUND(SUM(jp_sales), 2) AS jp_total,
        ROUND(SUM(pal_sales), 2) AS europe_total,
        ROUND(SUM(other_sales), 2) AS rest_of_world_total
    FROM '{csv_file}'
    WHERE total_sales IS NOT NULL
    GROUP BY genre
""").df()

print(kpi6.to_string())

               genre  na_total  jp_total  europe_total  rest_of_world_total
0         Simulation    152.84     35.98         86.86                24.34
1             Puzzle     64.35     28.70         25.78                 7.38
2          Adventure    157.26     45.64         91.52                30.11
3             Racing    269.66     20.10        179.10                56.43
4           Strategy     47.06     36.77         27.05                 7.29
5       Visual Novel      0.49      5.06          0.07                 0.13
6              Music     25.97      5.84         15.02                 4.93
7               Misc    295.66     55.47        146.50                58.52
8           Fighting    173.96     58.61         79.84                28.65
9          Education      0.60       NaN          0.20                 0.07
10        Board Game      0.06      0.04          0.22                 0.02
11           Shooter    528.22     33.87        324.72               108.42
12          

In [9]:
# Publisher Portfolio Efficiency
kpi7 = duckdb.sql(f"""
    SELECT 
        publisher,
        COUNT(title) AS total_releases,
        ROUND(SUM(total_sales), 2) AS global_revenue,
        ROUND(AVG(total_sales), 2) AS avg_revenue_per_title,
        ROUND(AVG(critic_score), 2) AS avg_critic_rating
    FROM '{csv_file}'
    WHERE total_sales IS NOT NULL
    GROUP BY publisher
    HAVING COUNT(title) >= 50
    ORDER BY avg_revenue_per_title DESC
    LIMIT 15
""").df()

print(kpi7.to_string())

                                 publisher  total_releases  global_revenue  avg_revenue_per_title  avg_critic_rating
0                           Rockstar Games              93          239.67                   2.58               8.51
1                       Bethesda Softworks             113          111.08                   0.98               7.43
2                                EA Sports             539          485.63                   0.90               7.88
3                                LucasArts             138          118.48                   0.86               7.18
4                          Electronic Arts             843          644.13                   0.76               7.38
5                               Activision            1043          722.77                   0.69               7.09
6                   Microsoft Game Studios              92           58.64                   0.64               7.66
7                 Warner Bros. Interactive             126      

In [10]:
# Platform Ecosystem Comparison
kpi8 = duckdb.sql(f"""
    SELECT 
        CASE 
            WHEN console IN ('PS', 'PS2', 'PS3', 'PS4', 'PS5', 'PSP', 'PSV') THEN 'Sony PlayStation'
            WHEN console IN ('XB', 'X360', 'XOne', 'XSX') THEN 'Microsoft Xbox'
            WHEN console IN ('N64', 'GC', 'Wii', 'WiiU', 'NS', 'GB', 'GBA', 'DS', '3DS') THEN 'Nintendo'
            WHEN console = 'PC' THEN 'PC'
            ELSE 'Other Hardware'
        END AS hardware_ecosystem,
        COUNT(title) AS total_catalog_size,
        ROUND(SUM(total_sales), 2) AS total_sales,
        ROUND(AVG(total_sales), 2) AS avg_sales_per_game
    FROM '{csv_file}'
    WHERE total_sales IS NOT NULL
    GROUP BY hardware_ecosystem
    ORDER BY total_sales DESC
""").df()

print(kpi8)

  hardware_ecosystem  total_catalog_size  total_sales  avg_sales_per_game
0   Sony PlayStation                7511      3256.71                0.43
1           Nintendo                6450      1543.15                0.24
2     Microsoft Xbox                2660      1360.41                0.51
3     Other Hardware                 690       267.78                0.39
4                 PC                1560       168.99                0.11
